## Custom SNN Python implementation for electronics engineering bachelor project

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

### Leaky integrate and fire neuron

In [ ]:
class LIF():
    """
        Simple Leaky Integrate-and-Fire (LIF) Neuron with STDP
    """
    def __init__(self, beta=0.9, threshold=2.0, reset=0.0, learning_rate=0.01):
        self.beta = beta                    # Decay factor
        self.threshold = threshold          # Spike threshold value - value that triggers a spike
        self.mem = 0                        # Membrane potential - current value
        self.reset = reset                  # Membrane potential reset value - value after spike
        self.last_spike = -1                # Time of last spike - counts time steps since last spike (-1 = no spike yet)
        self.learning_rate = learning_rate  # Learning rate for STDP
        self.weight = 0.2                   # Synaptic weight
        self.spk = 0                        # Spike
        

    def update(self, synaptic_input):
        # Compute decay, input and reset signals
        decay = (self.beta * self.mem)
        input = (synaptic_input * self.weight)
        self.reset = (self.spk * self.threshold)

        # Update membrane potential
        self.mem = decay + input - self.reset

        # If membrane potential exceeds threshold -> spike
        if self.mem > self.threshold:
            self.spk = 1
            self.last_spike = 0  # Reset last spike time

        # Return spike
        return self.spk
    
    
    def STDP(self, pre_spike):
        # Basic Spike-Timing-Dependent Plasticity (STDP)

        # Only update if both current and past neuron has fired
        if self.last_spike >= 0 and pre_spike >= 0:
            delta = self.last_spike - pre_spike

            if delta > 0:
                # Current fires after prev -> strengthen synapse
                self.weight += self.learning_rate
            else:
                # Current fires before prev -> weaken synapse
                self.weight -= self.learning_rate
    
    def rSTDP(self, prespike, reward):
        # STDP with reward modulation
        pass


In [31]:
# Simulation parameters
time_steps = 10

# Architecture parameter
input_size = 10     # Number of input neurons
output_size = 1     # Number of output neurons


# Instantiate layers as litst of neurons
input_neurons = [LIF(beta=0.9, threshold=1, learning_rate=0.01) for _ in range(input_size)]
output_neuron = [LIF(beta=0.9, threshold=1, learning_rate=0.01) for _ in range(output_size)]

print(f"Sucessfully created input layer with {len(input_neurons)}")
print(f"Sucessfully created output layer with {len(output_neuron)}")

# Instantiate synapses as a matrix
input_to_output_synapses = np.random.rand(input_size, output_size)

print(f"Sucessfully created synapse matrix with shape {input_to_output_synapses.shape}")


Sucessfully created input layer with 10
Sucessfully created output layer with 1
Sucessfully created synapse matrix with shape (10, 1)


In [32]:
# Simulation
for t in range(time_steps):
    # Generate random input spikes for input layer
    input_spikes = np.random.randint(0, 2, size=input_size)
    input_spikes = [0, 0, 1, 0, 1, 0, 1, 0, 0, 1]  # Example fixed input for testing

    # Update input layer neurons
    for i, neuron in enumerate(input_neurons):
        neuron_spike = neuron.update(input_spikes[i])

    # Update output layer neurons
    for j, neuron in enumerate(output_neuron):
        # Compute total synaptic input from input layer
        synaptic_input = sum(input_neurons[i].weight * input_spikes[i] for i in range(input_size))
        neuron_spike = neuron.update(synaptic_input)

    # Apply STDP learning rule
    for i, pre_neuron in enumerate(input_neurons):
        for j, post_neuron in enumerate(output_neuron):
            post_neuron.STDP(pre_neuron.last_spike)

    # Print spikes at current time step
    input_spike_str = ''.join(str(neuron.spk) for neuron in input_neurons)
    output_spike_str = ''.join(str(neuron.spk) for neuron in output_neuron)
    print(f"Time step {t}: Input spikes: {input_spike_str}, Output spike: {output_spike_str}")

Time step 0: Input spikes: 0000000000, Output spike: 0
Time step 1: Input spikes: 0000000000, Output spike: 0
Time step 2: Input spikes: 0000000000, Output spike: 0
Time step 3: Input spikes: 0000000000, Output spike: 0
Time step 4: Input spikes: 0000000000, Output spike: 0
Time step 5: Input spikes: 0000000000, Output spike: 0
Time step 6: Input spikes: 0010101001, Output spike: 0
Time step 7: Input spikes: 0010101001, Output spike: 0
Time step 8: Input spikes: 0010101001, Output spike: 0
Time step 9: Input spikes: 0010101001, Output spike: 1
